In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Find all CRMLSSold files inside the current notebooks folder
files = sorted(Path.cwd().rglob("CRMLSSold*.csv"))

# Exclude the combined 2022–2023 file
files = [
    file for file in files
    if "20220101_20231231" not in file.name
]

print("Files found:", len(files))
for file in files:
    print(file)

if len(files) == 0:
    raise FileNotFoundError(
        "No CSV files found. Check that the files are inside notebooks/data/"
    )

# Load each CSV file
frames = []

for file in files:
    temp = pd.read_csv(file, low_memory=False)
    temp["source_file"] = file.name
    frames.append(temp)

# Combine all monthly files
df = pd.concat(frames, ignore_index=True, sort=False)

# Convert CloseDate into a date format
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")

print("Raw shape:", df.shape)

Files found: 7
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202406_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202407_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202409.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202410.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202411.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202412.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202501_filled.csv
Raw shape: (154367, 81)


In [2]:
df = df[(df['PropertyType'] == 'Residential') &
        (df['PropertySubType'] == 'SingleFamilyResidence')].copy()

# These are the input columns the model will use to predict ClosePrice.
features = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
            'LotSizeSquareFeet', 'YearBuilt', 'DaysOnMarket', 'ListPrice',
            'Latitude', 'Longitude', 'City', 'CountyOrParish']
# ClosePrice is the value the model is trying to predict.
target = 'ClosePrice'
# Convert numeric predictors and the target from text to numbers.
for col in features + [target]:
    if col not in ['City', 'CountyOrParish']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
# Remove rows that do not have a valid positive sale price.
df = df[df[target].notna() & (df[target] > 0)].copy()
# Sort chronologically before removing duplicate listings.
df = df.sort_values(['CloseDate', 'ListingKey'])
# Remove duplicate records only when a valid ListingKey exists.
with_key = df[df['ListingKey'].notna()].drop_duplicates('ListingKey', keep='last')
# Preserve rows with no ListingKey instead of accidentally collapsing them together.
without_key = df[df['ListingKey'].isna()]
df = pd.concat([with_key, without_key], ignore_index=True)
# Create a YYYY-MM label used to build the time-based train/test split.
df['CloseMonth'] = df['CloseDate'].dt.to_period('M').astype(str)
print('Prepared rows:', len(df))
print('Available months:', sorted(df['CloseMonth'].dropna().unique()))

Prepared rows: 78580
Available months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01']


Missing-value strategy:
Rows with a missing or nonpositive target are removed because a supervised model cannot learn from them. Numeric predictors are median-imputed using the training data only. Categorical predictors are filled with Unknown and one-hot encoded. Numeric predictors are standardized inside the modeling pipeline.

In [4]:
# These are the numeric input columns used to predict ClosePrice.
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket",
    "ListPrice",
    "Latitude",
    "Longitude"
]

# These are categorical columns that need one-hot encoding.
categorical_features = [
    "City",
    "CountyOrParish"
]

# Apply different preprocessing steps to numeric and categorical columns.
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            # Replace missing numeric values with the training median.
            ("imputer", SimpleImputer(strategy="median")),

            # Put numeric variables on a similar scale.
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            # Replace missing categories with "Unknown".
            ("imputer", SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )),

            # Convert categories into numerical columns.
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

Time-based train/test split and training-window experiment

In [5]:
# Hold out the most recent available month as the test set.
test_month = '2025-01'
available_training_months = sorted(m for m in df['CloseMonth'].unique() if m < test_month)
results = []

# Try different training-window lengths to see which performs best.
for x in range(1, len(available_training_months) + 1):
    # Select the most recent x available months for training.
    train_months = available_training_months[-x:]
    train = df[df['CloseMonth'].isin(train_months)]
    test = df[df['CloseMonth'] == test_month]
    # Build one pipeline so preprocessing is learned only from the training data.
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ])
    # Fit the preprocessing steps and linear regression model on the training rows.
    model.fit(train[features], train[target])
    # Generate predictions for the held-out January test data.
    predictions = model.predict(test[features])
    results.append({
        'training_window_months': x,
        'training_months': ', '.join(train_months),
        'training_rows': len(train),
        'test_rows': len(test),
        # R-squared measures how much variation in the test prices the model explains.
        'test_r2': r2_score(test[target], predictions)
    })

window_results = pd.DataFrame(results).sort_values('test_r2', ascending=False)
display(window_results)
print('Selected window based on highest January test R2:',
      int(window_results.iloc[0]['training_window_months']))

,training_window_months,training_months,training_rows,test_rows,test_r2
5,6,"2024-06, 2024-07, 2024-09, 2024-10, 2024-11, 2...",70436,8144,0.651097
4,5,"2024-07, 2024-09, 2024-10, 2024-11, 2024-12",57899,8144,0.650980
3,4,"2024-09, 2024-10, 2024-11, 2024-12",44525,8144,0.650598
2,3,"2024-10, 2024-11, 2024-12",33630,8144,0.650431
1,2,"2024-11, 2024-12",21290,8144,0.650206
0,1,2024-12,10618,8144,0.647630


Selected window based on highest January test R2: 6


In [6]:
# Keep the cleaned fields needed for later modeling and documentation.
export_cols = ['ListingKey', 'CloseDate', 'CloseMonth', 'ClosePrice',
               'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
               'LotSizeSquareFeet', 'YearBuilt', 'DaysOnMarket', 'ListPrice',
               'Latitude', 'Longitude', 'City', 'CountyOrParish',
               'PropertyType', 'PropertySubType', 'source_file']
cleaned = df[export_cols].copy()
# Label January as test and all earlier available months as train.
cleaned['split'] = np.where(cleaned['CloseMonth'].eq(test_month), 'test', 'train')
# Save the cleaned dataset as a CSV file.
cleaned.to_csv('cleaned_california_single_family_sales.csv', index=False)
print('Saved', len(cleaned), 'rows to cleaned_california_single_family_sales.csv')

Saved 78580 rows to cleaned_california_single_family_sales.csv
